# Vulpes Flos 1.0: one training session

One 12-hour session of the Vulpes Flos 1.0 Base run (docs/KAGGLE.md in the repository). Every
session is a new version of this notebook: **Save Version -> Save & Run All**, then close the
browser. It finds the data and the newest checkpoint among the inputs, trains until ~45 min before
Kaggle's 12-hour limit, saves, and leaves the run in this notebook's output for the next session.

Settings (right-hand panel): **Accelerator: GPU T4 x2**, **Internet: on**, **Persistence: none**.
Inputs: the private dataset with the shards (K3) and, from the second session on, **this notebook's
own output** (Add Input -> Your Work -> this notebook). Change `COMMIT` only between runs.

In [ ]:
import time

SESSION_START = time.time()  # the 12-hour limit counts from here
REPO = "https://github.com/eyboss202/VulpesAI.git"
COMMIT = "main"  # pin a commit hash for the long run: every session of a run uses the same code
EXTRA = ""  # e.g. "--hours 2 max_steps=80" for K6's pilot; key=value overrides go to scripts/train.py

In [ ]:
import os
import subprocess

url = REPO
try:  # a private repository needs a GitHub token saved as the Kaggle secret GITHUB_TOKEN
    from kaggle_secrets import UserSecretsClient

    url = REPO.replace(
        "https://", f"https://x-access-token:{UserSecretsClient().get_secret('GITHUB_TOKEN')}@"
    )
except Exception:
    pass
CODE = "/tmp/VulpesAI"  # outside /kaggle/working, so the code is not saved as output
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--quiet", url, CODE], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "--quiet", COMMIT], check=True)
print(
    subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
)
# Kaggle's image already has PyTorch, NumPy, PyYAML, safetensors, tqdm and TensorBoard.
subprocess.run(["pip", "install", "--quiet", "--no-deps", "-e", CODE], check=True)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!cd /tmp/VulpesAI && python scripts/kaggle/session.py --started-at {SESSION_START} {EXTRA}